## Experiment 2: Partitioning Scheme Hyperparameter

How OOD classification quality depends on partitioning hyperparameters 
  * when only one scheme family is used at a time: Quantile Binning, KMeans, or Decision Tree.
  * when several schemas used

For every dataset we run:
1. Quantile-only experiment with dependency on `n_bins`.
2. KMeans-only experiment with dependency on `n_kmeans_clusters`.
3. DecisionTree-only experiment with dependencies on `tree_max_depth` and `n_tree_partitions`.
4. Quantile + KMeans + DecisionTree experiments with dependency on (`n_bins`, `n_kmeans_clusters`, `tree_max_depth`, `n_tree_partitions`)


In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display

from point_wise_partition_meta.partition_ood import load_dataset
from point_wise_partition_meta.partitioning_scheme_learning import (
    run_single_parameter_sweep,
    learn_all_schemes_hyperparameters_with_validation,
)
from partition_ood import DEFAULT_META_FEATURES

In [ ]:
TEST_SIZE = 0.3
VAL_SIZE = 0.3
RANDOM_STATE = 42
USE_RAW_FEATURES = True
CV_FOLDS = 5
BALANCE_STRATEGY = 'oversample'
CLASSIFIER_NAME = 'mlp'

META_FEATURES = [
    'dist_to_mean', 'normalized_dist', 'cosine_dist', 'dist_to_median', 'log_likelihood',
    'local_outlier_factor_score', 'knn_distance_k',
    'mean_norm', 'std_norm', 'skew_norm', 'kurtosis_norm', 'median_abs_deviation_norm',
    'iqr_norm', 'trimmed_mean_norm',
    'cell_entropy', 'marginal_entropy_mean', 'marginal_entropy_std',
    'marginal_kl_to_reference_mean', 'quantile_surprisal_mean',
    'mahalanobis', 'covariance_trace', 'covariance_logdet', 'mean_abs_correlation',
    'pairwise_pearson_corr_mean_abs', 'pairwise_spearman_corr_mean_abs',
    'covariance_condition_number',
    'pairwise_mutual_info_mean', 'pairwise_mutual_info_max', 'total_correlation',
    'joint_entropy_pairwise_mean', 'pairwise_js_divergence_mean',
    'log_count', 'density', 'out_of_range_count', 'n_beyond_2std',
    'partition_agreement_count', 'leaf_depth',
]

QUANTILE_GRID = [3, 5, 10, 12, 15]
KMEANS_GRID = [2, 4, 6, 8]
TREE_DEPTH_GRID = [2, 3, 5]
TREE_PARTITIONS_GRID = [3, 5, 10]

JOINT_SEARCH_GRID = {
    'n_bins': QUANTILE_GRID,
    'n_kmeans_clusters': KMEANS_GRID,
    'tree_max_depth': TREE_DEPTH_GRID,
    'n_tree_partitions': TREE_PARTITIONS_GRID,
}


In [6]:
def _plot_dependency(df, x_col, dataset_name, title):
    plt.figure(figsize=(8, 5))
    sns.lineplot(data=df, x=x_col, y='roc_auc', marker='o', linewidth=2)
    if 'roc_auc_std' in df.columns:
        low = df['roc_auc'] - df['roc_auc_std']
        high = df['roc_auc'] + df['roc_auc_std']
        plt.fill_between(df[x_col], low, high, alpha=0.2)
    plt.title(f'{dataset_name} | {title}')
    plt.xlabel(x_col)
    plt.ylabel('ROC-AUC (CV mean)')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def run_dataset_experiment(dataset_name, plot_dependencies=True):
    ID_data, OOD_data = load_dataset(dataset_name)

    quantile_df = run_single_parameter_sweep(
        ID_data=ID_data,
        OOD_data=OOD_data,
        scheme_type='quantile',
        parameter_name='n_bins',
        parameter_values=QUANTILE_GRID,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        use_raw_features=USE_RAW_FEATURES,
        meta_features=META_FEATURES,
        classifier_name=CLASSIFIER_NAME,
        balance_strategy=BALANCE_STRATEGY,
        cv_folds=CV_FOLDS,
    )

    kmeans_df = run_single_parameter_sweep(
        ID_data=ID_data,
        OOD_data=OOD_data,
        scheme_type='kmeans',
        parameter_name='n_kmeans_clusters',
        parameter_values=KMEANS_GRID,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        use_raw_features=USE_RAW_FEATURES,
        meta_features=META_FEATURES,
        classifier_name=CLASSIFIER_NAME,
        balance_strategy=BALANCE_STRATEGY,
        cv_folds=CV_FOLDS,
    )

    tree_depth_df = run_single_parameter_sweep(
        ID_data=ID_data,
        OOD_data=OOD_data,
        scheme_type='tree',
        parameter_name='tree_max_depth',
        parameter_values=TREE_DEPTH_GRID,
        fixed_params={'n_tree_partitions': 5},
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        use_raw_features=USE_RAW_FEATURES,
        meta_features=META_FEATURES,
        classifier_name=CLASSIFIER_NAME,
        balance_strategy=BALANCE_STRATEGY,
        cv_folds=CV_FOLDS,
    )

    tree_partitions_df = run_single_parameter_sweep(
        ID_data=ID_data,
        OOD_data=OOD_data,
        scheme_type='tree',
        parameter_name='n_tree_partitions',
        parameter_values=TREE_PARTITIONS_GRID,
        fixed_params={'tree_max_depth': 3},
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        use_raw_features=USE_RAW_FEATURES,
        meta_features=META_FEATURES,
        classifier_name=CLASSIFIER_NAME,
        balance_strategy=BALANCE_STRATEGY,
        cv_folds=CV_FOLDS,
    ) 

    learned_joint = learn_all_schemes_hyperparameters_with_validation(
        ID_data=ID_data,
        OOD_data=OOD_data,
        search_grid=JOINT_SEARCH_GRID,
        test_size=TEST_SIZE,
        val_size=VAL_SIZE,
        random_state=RANDOM_STATE,
        use_raw_features=USE_RAW_FEATURES,
        meta_features=META_FEATURES,
        classifier_name=CLASSIFIER_NAME,
        balance_strategy=BALANCE_STRATEGY,
    )

    print(f'Dataset: {dataset_name}')
    print('Best joint params:', learned_joint['best_params'])

    if plot_dependencies:
        _plot_dependency(
            quantile_df,
            'n_bins',
            dataset_name,
            'Quantile-only: ROC-AUC vs n_bins',
        )
        _plot_dependency(
            kmeans_df,
            'n_kmeans_clusters',
            dataset_name,
            'KMeans-only: ROC-AUC vs n_kmeans_clusters',
        )
        _plot_dependency(
            tree_depth_df,
            'tree_max_depth',
            dataset_name,
            'Tree-only: ROC-AUC vs tree_max_depth (n_tree_partitions=5)',
        )
        _plot_dependency(
            tree_partitions_df,
            'n_tree_partitions',
            dataset_name,
            'Tree-only: ROC-AUC vs n_tree_partitions (tree_max_depth=3)',
        )

    print('Quantile sweep')
    display(quantile_df)
    print('KMeans sweep')
    display(kmeans_df)
    print('Tree sweep by depth')
    display(tree_depth_df)
    print('Tree sweep by number of partitions')
    display(tree_partitions_df)

    learned_metrics_df = learned_joint['learned_metrics_df'][[
        'classifier',
        'n_bins',
        'n_kmeans_clusters',
        'tree_max_depth',
        'n_tree_partitions',
        'roc_auc',
        'pr_auc',
        'accuracy',
        'precision',
        'recall',
        'f1',
    ]]
    print('Learned hyperparameter test metrics')
    display(learned_metrics_df)

    learned_metrics_df = learned_metrics_df.copy()
    learned_metrics_df['dataset'] = dataset_name

    return {
        'quantile_sweep': quantile_df,
        'kmeans_sweep': kmeans_df,
        'tree_depth_sweep': tree_depth_df,
        'tree_partitions_sweep': tree_partitions_df,
        'learned_joint': learned_joint,
        'learned_rows': learned_metrics_df.to_dict(orient='records'),
    }


## Taxi


In [7]:
taxi_exp = run_dataset_experiment('Taxi')


KeyboardInterrupt: 

## Electricity


In [ ]:
electricity_exp = run_dataset_experiment('Electricity')


## Income


In [ ]:
income_exp = run_dataset_experiment('Income')


## MVx6


In [ ]:
mvx6_exp = run_dataset_experiment('MVx6')


## Diabetes


In [ ]:
diabetes_exp = run_dataset_experiment('Diabetes')


## California


In [ ]:
california_exp = run_dataset_experiment('California')


## ACS Accidents


In [ ]:
acs_exp = run_dataset_experiment('ACS Accidents')


In [ ]:
all_learned = []
for obj in [
    taxi_exp,
    electricity_exp,
    income_exp,
    mvx6_exp,
    diabetes_exp,
    california_exp,
    acs_exp,
]:
    all_learned.extend(obj['learned_rows'])

summary_df = pd.DataFrame(all_learned)
display(summary_df)

pivot_roc = summary_df.pivot(index='dataset', columns='scheme', values='roc_auc')
display(pivot_roc)

plt.figure(figsize=(9, 5))
sns.heatmap(pivot_roc, annot=True, fmt='.3f', cmap='YlGnBu')
plt.title('Learned hyperparameters: ROC-AUC by dataset and scheme')
plt.tight_layout()
plt.show()
